In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("ollama:qwen2.5:3b")
standard_model = init_chat_model("ollama:qwen2.5:3b")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:qwen2.5:3b",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I'm sorry, but as an AI assistant, I don't have the ability to perform physical tasks like watering plants. My main function is to provide information and help with tasks that require digital assistance. However, if you need a reminder or someone to remind you to water the plant, I'd be happy to suggest setting up a notification or task for that purpose!


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

qwen2.5:3b


In [7]:

from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

That depends on several factors, such as the size of the current pot and how much space you have for it in your office. If it's already quite large but you need more room or if the soil needs changing due to nutrient depletion, then a replacement might be considered after a few years. Would you like me to look up specific care guidelines for your plant?


In [8]:
print(response["messages"][-1].response_metadata["model_name"])

qwen2.5:3b
